## Setup


In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import concurrent.futures
import matplotlib.ticker as mtick
import seaborn as sns
import re
import time
import functools

from datasets import Dataset, load_dataset, load_from_disk
from geopy.geocoders import Nominatim
from timezonefinder import TimezoneFinder
from tqdm.notebook import tqdm
from chromadb.utils import embedding_functions
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import cohen_kappa_score
from sentence_transformers import SentenceTransformer
from pathlib import Path
from openai import OpenAI
from transformers import pipeline

from src.config import (
    MAX_WORKERS,
    WAIT_TIME,
    RANDOM_SEED,
    BUFFER_SIZE,
    SAMPLE_PERCENTAGE,
    VALIDATION_SAMPLE,
    CHECKPOINT_INTERVAL,
    BATCH_SIZE,
    MODEL_ID,
    OPENAI_API_KEY,
    OPENROUTER_API_KEY,
    OPENROUTER_BASE_URL,
    WILDCHAT_DATASET,
    CONVERSATION_FIELD,
    LANGUAGE_FIELD,
    TARGET_LANGUAGE,
    PARENT_COLUMN,
    COUNT_VARIABLE,
    MAJOR_CATEGORIES,
    DATA_PATH,
    INPUT_DATA_DIR,
    OUTPUT_DIR,
    LABELED_DATA_DIR,
    PROMPTS_DIR,
    BATCHES_DIR,
    INPUT_BATCHES_DIR,
    OUTPUT_BATCHES_DIR,
    RAW_1P_DATA_PATH,
    RAW_CLAUDE_DATA_PATH,
    TASK_STATEMENTS_PATH,
    TASK_RATING_PATH,
    WILDCHAT_FULL,
    WILDCHAT_ENGLISH,
    WILDCHAT_SAMPLES_FILE,
    TASK_MAPPING_PROMPT_PATH,
    LABOR_TRANSFER_PROMPT_PATH,
    TASK_MAPPING_OUTPUT_PATH,
    TIMEZONES_OUTPUT_PATH,
    LABOR_TRANSFER_OUTPUT_FILE,
    LABOR_TRANSFER_BATCH_FILE,
    LABOR_TRANSFER_BATCH_OUTPUT_FILE,
    WORK_RELATED_BATCH_FILE,
    WORK_RELATED_BATCH_OUTPUT_FILE,
    WORK_RELATED_PROMPT_PATH,
    WORK_RELATED_OUTPUT_PATH,
)
from src.batch_utils import (
    create_batch_file,
    submit_batch_file,
    retrieve_batch_job_results,
    submit_and_retrieve,
)
from src.llm_utils import (
    RobustEncoder,
    get_messages,
    build_messages,
    strip_messages,
    format_conversation,
    get_gpt_response,
    parallelize_llm_call,
    format_tasks,
    format_options,
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

submit_batch_file = functools.partial(submit_batch_file, client=client)
retrieve_batch_job_results = functools.partial(
    retrieve_batch_job_results, client=client
)
submit_and_retrieve = functools.partial(submit_and_retrieve, client=client)
get_gpt_response = functools.partial(get_gpt_response, client=client)
parallelize_llm_call = functools.partial(parallelize_llm_call, client=client)

In [ ]:
if not WILDCHAT_FULL.exists():
    wildchat_ds = load_dataset(
        path=WILDCHAT_DATASET,
        split="train",
    )
    wildchat_ds.save_to_disk(WILDCHAT_FULL)

if not WILDCHAT_ENGLISH.exists():
    wildchat_ds = load_from_disk(WILDCHAT_FULL)
    english_conversations_rows = wildchat_ds.filter(
        lambda x: x[LANGUAGE_FIELD] == TARGET_LANGUAGE
    )
    english_conversations_rows.save_to_disk(WILDCHAT_ENGLISH)
else:
    english_conversations_rows = load_from_disk(WILDCHAT_ENGLISH)

total_rows = len(english_conversations_rows)
sample_size = int(SAMPLE_PERCENTAGE * total_rows)
sample_conversations_rows = (
    english_conversations_rows.shuffle(
        seed=RANDOM_SEED,
    )
    .select(range(sample_size))
    .to_pandas()
)
print(f"Total conversations: {total_rows}, sample size: {sample_size}")

Loading dataset from disk:   0%|          | 0/87 [00:00<?, ?it/s]

Filter:   0%|          | 0/3199860 [00:00<?, ? examples/s]

: 

In [4]:
sample_conversations_rows.shape

(50381, 14)

In [5]:
sample_conversations_rows.columns.values

array(['conversation_hash', 'model', 'timestamp', 'conversation', 'turn',
       'language', 'openai_moderation', 'detoxify_moderation', 'toxic',
       'redacted', 'state', 'country', 'hashed_ip', 'header'],
      dtype=object)

In [6]:
wildchat_columns_to_keep = [CONVERSATION_FIELD, "timestamp", "country", "state"]
sample_conversations = sample_conversations_rows[wildchat_columns_to_keep].copy()
sample_conversations.shape

(50381, 4)

In [8]:
device = ""

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [10]:
client = OpenAI(
    api_key=OPENAI_API_KEY,
)

### Batching configuration

This subsection contains configuration related to the batch processing of conversations. The way it works is:

1. `create_batch_file` creates a batch file containing multiple conversations, each formatted as a separate request to the OpenAI API. The batch file is saved in `WORK_RELATED_BATCH_FILE`.
2. `submit_batch_file` submits the batch file to the OpenAI API for processing, and returns a `batch_job_id` that can be used to track the job.
3. `retrieve_batch_job_results` takes the `batch_job_id` and retrieves the results of the batch processing, saving them to `WORK_RELATED_BATCH_OUTPUT_FILE`.
   It's up to the functions that call these to handle the formatting of the input conversations and the parsing of the output results.


## Conversation-task mapping

In order to map conversations to tasks, we are using the following steps:

1. Retrieve the conversations from Wildchat, sampling for English conversations
2. Filter out all the non-work-related conversations
3. Map the conversations to tasks using a prompt that asks the model to identify the task associated with each conversation.


### Work-related conversations filtering


In [ ]:
def parse_work_related_batch_results(output_path: Path):
    results = {}
    with open(output_path, "r") as f:
        for line in f:
            record = json.loads(line)

            if record.get("error") is not None:
                results[record["custom_id"]] = "ERROR"
                continue

            custom_id = record["custom_id"]
            content = record["response"]["body"]["choices"][0]["message"]["content"]

            try:
                parsed_content = json.loads(content)

                if "answer" in parsed_content:
                    answer = parsed_content["answer"]

                elif (
                    "command" in parsed_content and "args" in parsed_content["command"]
                ):
                    answer = parsed_content["command"]["args"].get("message", "Unknown")

                else:
                    raise ValueError("Not valid JSON format")

            except (KeyError, ValueError, json.JSONDecodeError):
                print(
                    f"Wrong JSON format for {custom_id}, trying to retrieve the answer from the text..."
                )
                if re.search(r"\b(No|no)\b", content):
                    answer = "No"
                elif re.search(r"\b(Yes|yes)\b", content):
                    answer = "Yes"
                else:
                    print(
                        f"It was not possible to retrieve the answer for {custom_id}. Full content: {content}"
                    )
                    answer = "Unknown"

            if isinstance(answer, str):
                answer = answer.strip(" .,\n").capitalize()

            results[custom_id] = answer

    return results


def filter_work_conversations(conversations: pd.DataFrame, path: Path) -> list:
    """
    Filter conversations to determine which ones are work-related using the GPT model.

    :param conversations: DataFrame containing conversations to filter
    :param path: Path to save the filtered conversations
    :return: list containing boolean values indicating whether each conversation is work-related or not
    """
    formatted_messages = []
    template = get_messages(path=WORK_RELATED_PROMPT_PATH)
    word_id = path.stem.split("_")[0]

    for index, row in conversations.iterrows():
        raw_conversation = row["conversation"]
        raw_conversation = strip_messages(raw_conversation)
        conversation = format_conversation(conversation=raw_conversation)
        messages = build_messages(template=template, conversation=conversation)
        formatted_messages.append(messages)

    results = submit_and_retrieve(
        messages=formatted_messages,
        batch_file=WORK_RELATED_BATCH_FILE,
        output_file=WORK_RELATED_BATCH_OUTPUT_FILE,
        word_id=word_id,
        parse_func=parse_work_related_batch_results,
    )
    answers = [results.get(f"{word_id}_{i}") for i in range(len(formatted_messages))]
    return answers

In [ ]:
if not WORK_RELATED_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        conversations=sample_conversations, path=WORK_RELATED_OUTPUT_PATH
    )
    work_related_conversations_df = sample_conversations.copy()
    work_related_conversations_df["conversation"] = work_related_conversations_df[
        "conversation"
    ].apply(lambda x: [dict(msg) for msg in x] if not isinstance(x, list) else x)
    work_related_conversations_df["is_work_related"] = answers
    work_related_conversations_df["conversation"] = work_related_conversations_df[
        "conversation"
    ].apply(lambda x: json.dumps(x, cls=RobustEncoder))
    work_related_conversations_df["timestamp"] = work_related_conversations_df[
        "timestamp"
    ].apply(lambda x: x.isoformat() if hasattr(x, "isoformat") else str(x))
    work_related_conversations_df["conversation"] = work_related_conversations_df[
        "conversation"
    ].apply(lambda conv: strip_messages(conv) if isinstance(conv, list) else None)
    work_related_conversations_df.to_csv(WORK_RELATED_OUTPUT_PATH, index=False)
else:
    work_related_conversations_df = pd.read_csv(WORK_RELATED_OUTPUT_PATH)
work_related_conversations_df.head(1)

conversation  \
0  [{'role': 'user', 'content': 'can you name any famous artist relevant to multi section lunch boxes '}, {'role': 'assistant', 'content': "While no specific artist is primarily known for their work on multi-section lunch boxes, various contemporary artists and designers have created innovative and artistic designs for lunch boxes and similar items, blending functionality with aesthetics. \n\nOne notable figure is **Katherine Hamnett**, a British fashion designer known for her bold designs and use of sustainable materials. While she may not specifically focus on lunch boxes, her emphasis on eco-friendly products could extend to reusable, multi-section lunch containers.\n\nAdditionally, **Martha Stewart** has a reputation for home goods and kitchen items, including stylish lunch boxes and containers that often embody a creative and artistic flair.\n\nFor more niche artistic interpretations, many artists collaborate with brands to create limited-edition lunch boxes featuring their artwork, particularly in popular culture, anime, or children's designs.\n\nIf you're looking for artists who generally create functional art or who have made a name in product design, names like **Tinkerbell Sisters**, known for their cute and whimsical designs, might come to mind, especially for children's items. \n\nHowever, it's worth noting that many modern artists and designers may engage with the functionality and aesthetics of everyday items, including lunch boxes, in various ways."}, {'role': 'user', 'content': 'Can you make a design and technology design movement research about modernism and answer these questions Designer?\n\nWho is the designer, and what do you know about their background?\n\n2. What’s Their Style?\n\nHow would you describe the designer’s style? (e.g., modern, simple, colourful)\n\n3. What Makes Their Designs Unique?\n\nWhat are some special features or details that make their products stand out?\n\n4. What Materials Do They Use?\n\nWhat kinds of materials or technologies does the designer use in their products?\n\n5. What’s a Famous Product?\n\nPick a well-known product by this designer. What makes this product special?\n\n6. What Are the Pros and Cons?\n\nWhat do you think is great about the designer’s work? Is there anything you think\ncould be better?\n\n7. What have you learned ?\n\nHow would you use this information when you are designing?\n'}, {'role': 'assistant', 'content': "### Design and Technology Design Movement Research: Modernism\n\n**Designer**: Marcel Breuer\n\n**1. Who is the designer, and what do you know about their background?**\n\nMarcel Breuer (1902-1981) was a Hungarian-born modernist architect and furniture designer who was influential in the field of modern design. He studied at the Bauhaus school in Germany, where he was influenced by the principles of functionalism and simplicity. Breuer worked under Walter Gropius, the founder of the Bauhaus, and later became a prominent figure in his own right. He is best known for his innovative use of materials and his contribution to the development of modern architecture and furniture design.\n\n**2. What’s their style?**\n\nBreuer's style can be described as modern and functional, characterized by clean lines, geometric forms, and the innovative use of materials. He embraced minimalist principles, focusing on the function of objects while maintaining an aesthetic appeal. His designs often exhibit a sense of balance and proportion, with a commitment to creating forms that were practical yet visually engaging.\n\n**3. What makes their designs unique?**\n\nBreuer’s designs are unique for their sculptural quality and the use of new materials for furniture, particularly tubular steel. His furniture often has an industrial yet elegant appearance, where the structure itself becomes a design feature. His approach not only served the functional needs but also created visually striking pieces. Breuer’s ability to combine artistry with utility helped t

In [ ]:
work_related_conversations_df = (
    work_related_conversations_df.merge(
        sample_conversations[["conversation", "country", "state"]],
        left_index=True,
        right_index=True,
        how="inner",
    )
    .drop(columns=["conversation_y"])
    .rename(columns={"conversation_x": "conversation"})
)
work_related_conversations_df.shape

(50381, 5)

In [ ]:
work_related_conversations_df = work_related_conversations_df[
    work_related_conversations_df["is_work_related"].str.strip() == "Yes"
]
work_related_conversations_df.shape

(36680, 5)

In [33]:
work_related_conversations_df.head(1)

conversation  \
0  [{'role': 'user', 'content': 'can you name any famous artist relevant to multi section lunch boxes '}, {'role': 'assistant', 'content': "While no specific artist is primarily known for their work on multi-section lunch boxes, various contemporary artists and designers have created innovative and artistic designs for lunch boxes and similar items, blending functionality with aesthetics. \n\nOne notable figure is **Katherine Hamnett**, a British fashion designer known for her bold designs and use of sustainable materials. While she may not specifically focus on lunch boxes, her emphasis on eco-friendly products could extend to reusable, multi-section lunch containers.\n\nAdditionally, **Martha Stewart** has a reputation for home goods and kitchen items, including stylish lunch boxes and containers that often embody a creative and artistic flair.\n\nFor more niche artistic interpretations, many artists collaborate with brands to create limited-edition lunch boxes featuring their artwork, particularly in popular culture, anime, or children's designs.\n\nIf you're looking for artists who generally create functional art or who have made a name in product design, names like **Tinkerbell Sisters**, known for their cute and whimsical designs, might come to mind, especially for children's items. \n\nHowever, it's worth noting that many modern artists and designers may engage with the functionality and aesthetics of everyday items, including lunch boxes, in various ways."}, {'role': 'user', 'content': 'Can you make a design and technology design movement research about modernism and answer these questions Designer?\n\nWho is the designer, and what do you know about their background?\n\n2. What’s Their Style?\n\nHow would you describe the designer’s style? (e.g., modern, simple, colourful)\n\n3. What Makes Their Designs Unique?\n\nWhat are some special features or details that make their products stand out?\n\n4. What Materials Do They Use?\n\nWhat kinds of materials or technologies does the designer use in their products?\n\n5. What’s a Famous Product?\n\nPick a well-known product by this designer. What makes this product special?\n\n6. What Are the Pros and Cons?\n\nWhat do you think is great about the designer’s work? Is there anything you think\ncould be better?\n\n7. What have you learned ?\n\nHow would you use this information when you are designing?\n'}, {'role': 'assistant', 'content': "### Design and Technology Design Movement Research: Modernism\n\n**Designer**: Marcel Breuer\n\n**1. Who is the designer, and what do you know about their background?**\n\nMarcel Breuer (1902-1981) was a Hungarian-born modernist architect and furniture designer who was influential in the field of modern design. He studied at the Bauhaus school in Germany, where he was influenced by the principles of functionalism and simplicity. Breuer worked under Walter Gropius, the founder of the Bauhaus, and later became a prominent figure in his own right. He is best known for his innovative use of materials and his contribution to the development of modern architecture and furniture design.\n\n**2. What’s their style?**\n\nBreuer's style can be described as modern and functional, characterized by clean lines, geometric forms, and the innovative use of materials. He embraced minimalist principles, focusing on the function of objects while maintaining an aesthetic appeal. His designs often exhibit a sense of balance and proportion, with a commitment to creating forms that were practical yet visually engaging.\n\n**3. What makes their designs unique?**\n\nBreuer’s designs are unique for their sculptural quality and the use of new materials for furniture, particularly tubular steel. His furniture often has an industrial yet elegant appearance, where the structure itself becomes a design feature. His approach not only served the functional needs but also created visually striking pieces. Breuer’s ability to combine artistry with utility helped t

In [ ]:
def get_timezone_for_location(
    state: str, country: str, geolocator: Nominatim, tf: TimezoneFinder
) -> list[str]:
    try:
        query = f"{state}, {country}" if pd.notnull(state) else country
        location = geolocator.geocode(query)
        print(f"Geocoding query: '{query}' -> Location: {location}", flush=True)
        if location:
            print(
                f"Found location for query '{query}': {location.address} (lat: {location.latitude}, lng: {location.longitude})",
                flush=True,
            )
            return tf.timezone_at(lng=location.longitude, lat=location.latitude)
    except Exception as e:
        print(
            f"Error geocoding location for state '{state}' and country '{country}': {e}",
            flush=True,
        )


def find_timezones(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find timezones for locations in the DataFrame.

    :param df: Input DataFrame containing location information.
    :return: DataFrame with an additional column 'timezone' containing the found timezones.
    """

    unique_locations = df[["state", "country"]].drop_duplicates()
    timezones_df = pd.DataFrame(columns=["state", "country", "timezone"])
    tf = TimezoneFinder(in_memory=True)
    geolocator = Nominatim(user_agent="chatgpt_at_work_project")

    for _, row in unique_locations.iterrows():
        state = row["state"]
        country = row["country"]
        timezone = get_timezone_for_location(
            state=state, country=country, geolocator=geolocator, tf=tf
        )
        time.sleep(1)  # Sleep to respect geocoding service rate limits
        record = {"state": state, "country": country, "timezone": timezone}
        timezones_df = pd.concat(
            [timezones_df, pd.DataFrame([record])], ignore_index=True
        )

    df = df.merge(timezones_df, on=["state", "country"], how="left")
    return df

In [35]:
if not TIMEZONES_OUTPUT_PATH.exists():
    timezones_df = find_timezones(df=work_related_conversations_df)
    timezones_df.to_csv(TIMEZONES_OUTPUT_PATH, index=False)
else:
    timezones_df = pd.read_csv(TIMEZONES_OUTPUT_PATH)
timezones_df.shape

(36680, 6)

In [ ]:
work_related_conversations_df = timezones_df.copy()
work_related_conversations_df = work_related_conversations_df[
    work_related_conversations_df["timezone"].notnull()
]
work_related_conversations_df.shape

(36131, 6)

In [ ]:
def normalize_timezone(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize timezone by converting UTC to the local timezone.

    :param df: Input DataFrame containing a 'timezone' column to normalize.
    :return: DataFrame with an additional column 'normalized_timezone' containing the normalized timezone names.
    """
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize("UTC")

    df["timestamp_local"] = df.apply(
        lambda row: row["timestamp"].tz_convert(row["timezone"]), axis=1
    )
    return df

In [38]:
work_related_conversations_df = normalize_timezone(df=work_related_conversations_df)
work_related_conversations_df.shape

(36131, 7)

In [260]:
work_related_conversations_df.columns

Index(['conversation', 'timestamp', 'is_work_related', 'country', 'state',
       'timezone', 'timestamp_local'],
      dtype='object')

In [ ]:
# TODO: Update with the new prompt that uses "Task ID, Job Title, Task" instead of Anthropic's hierarchy


def parse_task_mapping_batch_results(
    output_path: Path, n_batches: int
) -> dict[str, str]:
    results = {}

    for i in range(n_batches):
        current_path = output_path.with_stem(f"{output_path.stem}_{i}")
        if not current_path.exists():
            continue
        with open(current_path, "r") as f:
            for line in f:
                record = json.loads(line)

                if record.get("error") is not None:
                    results[record["custom_id"]] = "ERROR"
                    continue

                custom_id = record["custom_id"]
                content = record["response"]["body"]["choices"][0]["message"]["content"]

                # if isinstance(content, str):
                # content = ast.literal_eval(content)

                answer = "Unknown"
                try:
                    parsed_content = json.loads(content)

                    if "answer" in parsed_content:
                        answer = parsed_content["answer"]
                    elif (
                        "command" in parsed_content
                        and "args" in parsed_content["command"]
                    ):
                        content = parsed_content["command"]["args"]
                        if isinstance(content, dict):
                            answer = content.get("message", "Unknown")
                        else:
                            answer = "Unknown"
                except (KeyError, ValueError, json.JSONDecodeError):
                    print(
                        f"Wrong JSON format for {custom_id}, content: {content[:100]}"
                    )

                results[custom_id] = answer

    return results


def map_conversation_to_task(
    conversations: pd.DataFrame, tasks: pd.DataFrame, path: Path
) -> pd.DataFrame:
    """
    Maps user conversations to hierarchical tasks at multiple levels (levels 2, 1, and 0). The function takes in a
    DataFrame of conversations, a DataFrame of tasks containing hierarchical task definitions, and a file path where
    the resulting mappings will be saved. It processes tasks hierarchy starting from level 2 (high-level) tasks,
    then moves to level 1 (medium-level), and finally to level 0 (low-level) tasks. For each conversation, tasks
    are determined progressively based on parent-child relationships within the tasks DataFrame.

    :param conversations: A pandas DataFrame containing conversations. Each row should represent a conversation.
    :param tasks: A pandas DataFrame containing hierarchical task definitions. Each task should include information
                  about its level (0, 1, 2) and its parent task association if applicable.
    :param path: A Path object specifying the file path to save the mapping of conversations to tasks.
    :return: A pandas DataFrame containing the original conversations and their corresponding mapped hierarchical
             tasks at levels 2, 1, and 0.
    """
    template = get_messages(path=TASK_MAPPING_PROMPT_PATH)
    conversations_list = conversations["conversation"].tolist()
    n_batches = len(conversations_list) // BATCH_SIZE + 1

    result = pd.DataFrame({"conversation": conversations_list, "": ""})
    result.to_csv(path, index=False)
    return result

In [ ]:
if not TASK_MAPPING_OUTPUT_PATH.exists():
    mapped_conversations_df = map_conversation_to_task(
        conversations=work_related_conversations_df,
        tasks=tasks_df,
        path=TASK_MAPPING_OUTPUT_PATH,
    )
else:
    mapped_conversations_df = pd.read_csv(TASK_MAPPING_OUTPUT_PATH)
mapped_conversations_df = format_tasks(
    df=mapped_conversations_df, column_name="level_0_task"
)
mapped_conversations_df.shape

(50381, 4)

## Labor transfer

In this section, we analyze how ChatGPT transfers work from professionals to users.


In [ ]:
tasks_grouped = (
    tasks_df.groupby("task")["title"]
    .apply(lambda x: ", ".join(x.unique()))
    .reset_index()
)

roles_titles_df = roles_df.merge(
    tasks_grouped, left_on="level_0_task", right_on="task", how="left"
)
roles_titles_df.drop(columns=["task"], inplace=True)
roles_titles_df.shape

(18037, 19)

In [352]:
sample_50 = roles_titles_df.sample(50, random_state=RANDOM_SEED)
conversations_sample_50 = sample_50[["conversation", "level_0_task", "title"]]
conversations_sample_50.to_csv("conversations_sample_50.csv", index=False)

In [ ]:
open_router_model = "meta-llama/llama-3.3-70b-instruct:free"

open_router_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

In [ ]:
def parse_labor_transfer_batch_results(output_path: Path) -> dict:
    results = {}
    with open(output_path, "r") as f:
        for line in f:
            record = json.loads(line)
            custom_id = record["custom_id"]

            if record.get("error") is not None:
                results[custom_id] = None
                continue

            content = record["response"]["body"]["choices"][0]["message"]["content"]

            try:
                parsed_content = json.loads(content)
                label = parsed_content.get("label", "")
                results[custom_id] = label
            except (json.JSONDecodeError, KeyError, TypeError) as e:
                print(f"Error parsing {custom_id}: {e}")
                results[custom_id] = None

            if "LT0" in content:
                results[custom_id] = "LT0"
            elif "LT1" in content:
                results[custom_id] = "LT1"
            elif "LT2" in content:
                results[custom_id] = "LT2"

    return results


def analyze_labor_transfer(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyzes the potential labor transfer between two specified roles by comparing the conversations associated with each role.
    The function identifies conversations that are similar between the two roles based on their content and calculates a similarity score.

    :param df: DataFrame containing the conversations and their associated roles
    :param source_role: The name of the role from which labor transfer is being analyzed
    :param target_role: The name of the role to which labor transfer is being analyzed
    :return: A DataFrame containing pairs of conversations from the source and target roles along with their similarity scores
    """
    template = get_messages(path=LABOR_TRANSFER_PROMPT_PATH)
    formatted_messages = []
    word_id = "labor_transfer"

    for idx, row in df.iterrows():
        conversation = row["conversation"]
        title = row["title"]
        task = row["level_0_task"]
        messages = build_messages(
            template=template, conversation=conversation, title=title, task=task
        )
        formatted_messages.append(messages)

    results = submit_and_retrieve(
        messages=formatted_messages,
        batch_file=LABOR_TRANSFER_BATCH_FILE,
        output_file=LABOR_TRANSFER_BATCH_OUTPUT_FILE,
        word_id=word_id,
        parse_func=parse_labor_transfer_batch_results,
    )

    labels = [
        results.get(f"patents_{i}", "Unknown").strip()
        for i in range(len(formatted_messages))
    ]

    return labels

In [355]:
if not LABOR_TRANSFER_OUTPUT_FILE.exists():
    labor_transfer_labels = analyze_labor_transfer(
        df=sample_50,
    )
    labor_transfer_df = pd.DataFrame({"label": labor_transfer_labels})
    labor_transfer_df.to_csv(LABOR_TRANSFER_OUTPUT_FILE, index=False)
else:
    labor_transfer_df = pd.read_csv(LABOR_TRANSFER_OUTPUT_FILE)
labor_transfer_df.shape

Submitting messages for labor_transfer_0.03_output: total 50 messages in 1 batches of up to 5000 messages each
[] batches already exist. Skipping those batches and retrieving results from existing files.
Submitted batch 0 (50 requests): batch_6a0719cd9a788190acd9e452273b5a4f
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status validating. Waiting for 60 seconds before retrying...
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status validating. Waiting for 60 seconds before retrying...
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status in_progress. Waiting for 60 seconds before retrying...
Batch job batch_6a0719cd9a788190acd9e452273b5a4f is still in status in_progress. Waiting for 60 seconds before retrying...


(50, 1)

In [366]:
roles_df.columns

Index(['conversation', 'timestamp', 'is_work_related', 'level_2_task',
       'level_1_task', 'level_0_task', 'Function', 'orig_index', 'interaction',
       'social_dimensions', 'role_sentence', 'role', 'role_core',
       'embedding_role_core', 'cluster_id', 'emotion', 'timestamp_local',
       'hour'],
      dtype='object')

In [ ]:
test = roles_df[roles_df["role"] == "Code reviewer"]

In [ ]:
batch_output_file = "data/batches/output/labor_transfer_0.03_output_0.jsonl"
csv_sample_path = "conversations_sample_50.csv"

parsed_results = []

with open(batch_output_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        try:
            data = json.loads(line)
            custom_id = data.get("custom_id", "")

            row_idx = int(custom_id.split("_")[-1])

            choices = data.get("response", {}).get("body", {}).get("choices", [])
            if not choices:
                continue

            content_str = choices[0].get("message", {}).get("content", "{}")
            content_dict = json.loads(content_str)

            parsed_results.append(
                {
                    "original_index": row_idx,
                    "label": content_dict.get("label"),
                    "lt1_reason": content_dict.get("lt1_reason"),
                    "task_match": content_dict.get("task_match"),
                }
            )

        except Exception as e:
            print(f"Errore nel parsing della riga con custom_id '{custom_id}': {e}")

df_parsed = pd.DataFrame(parsed_results)

df_sample_50 = pd.read_csv(csv_sample_path)

df_final = df_sample_50.merge(
    df_parsed, left_index=True, right_on="original_index", how="left"
)

df_final = df_final.drop(columns=["original_index"]).reset_index(drop=True)

df_final.to_csv("labor_transfer_analysis.csv", index=False)

display(df_final.head(1))

conversation  \
0  [{'role': 'user', 'content': 'que opinas de este codigo y calidad?\nimport asyncio\nimport aiohttp\nfrom bs4 import BeautifulSoup\nimport json\nfrom typing import List, Dict, Optional\nimport ssl\nimport logging\nfrom datetime import datetime\nimport colorlog\n\n# Configure logging\nhandler = colorlog.StreamHandler()\nhandler.setFormatter(colorlog.ColoredFormatter(\n    \'%(log_color)s%(asctime)s [%(levelname)s] %(message)s\',\n    datefmt=\'%Y-%m-%d %H:%M:%S\',\n    log_colors={\n        \'DEBUG\':    \'light_cyan\',\n        \'INFO\':     \'light_green\',\n        \'WARNING\':  \'light_yellow\',\n        \'ERROR\':    \'light_red\',\n        \'CRITICAL\': \'light_red,bg_white\',\n    }\n))\n\nlogger = colorlog.getLogger(__name__)\nlogger.addHandler(handler)\nlogger.setLevel(logging.INFO)\n\n# Remove any existing handlers to avoid duplicate logs\nfor hdlr in logger.handlers[:-1]:\n    logger.removeHandler(hdlr)\n\n# Configuration settings\nCONFIG = {\n    \'base_url\': \'https://10.0.0.2\',\n    \'username\': \'misael_campos\',\n    \'password\': \'d337O8AetR3p4\',\n    \'output_json\': \'aggregated_pon_onu_data.json\',\n    \'post_endpoint\': \'/action/pononuopticalinfo.html\',\n    \'who\': 100,\n    \'onuid\': 0,\n    \'pon_range\': range(1, 17),\n    \'onu_group_range\': range(0, 2),\n    \'request_timeout\': aiohttp.ClientTimeout(\n        total=240,      # Total operation timeout\n        connect=120,   # Connection timeout\n        sock_read=120  # Socket read timeout\n    ),\n    \'max_concurrent_requests\': 8,  # Reduced to prevent server overload\n    \'max_retries\': 3,              # Maximum number of retry attempts\n    \'retry_delay\': 1,              # Delay between retries in seconds\n    \'batch_size\': 8                # Number of concurrent requests per batch\n}\n\nclass AsyncScraper:\n    def __init__(self, base_url: str):\n        self.base_url = base_url.rstrip(\'/\')\n        self.ssl_context = ssl.create_default_context()\n        self.ssl_context.check_hostname = False\n        self.ssl_context.verify_mode = ssl.CERT_NONE\n        self.session = None\n        self.sem = asyncio.Semaphore(CONFIG[\'max_concurrent_requests\'])\n\n    async def __aenter__(self):\n        timeout = CONFIG[\'request_timeout\']\n        self.session = aiohttp.ClientSession(\n            connector=aiohttp.TCPConnector(\n                ssl=self.ssl_context,\n                limit=CONFIG[\'max_concurrent_requests\']\n            ),\n            timeout=timeout,\n            headers={\'User-Agent\': \'Mozilla/5.0 (Windows NT 10.0; Win64; x64)\'}\n        )\n        return self\n\n    async def __aexit__(self, exc_type, exc_val, exc_tb):\n        if self.session:\n            await self.session.close()\n\n    async def login(self, username: str, password: str) -> bool:\n        login_url = f"{self.base_url}/action/main.html"\n        login_data = {\n            \'user\': username,\n            \'pass\': password,\n            \'button\': \'Login\',\n            \'who\': \'100\',\n        }\n        \n        for attempt in range(CONFIG[\'max_retries\']):\n            try:\n                async with self.session.post(login_url, data=login_data) as response:\n                    text = await response.text()\n                    if "Sorry, you do not have access" in text:\n                        logger.error("Login failed: Access denied")\n                        return False\n                    elif "Error" in text or "Incorrect password" in text:\n                        logger.error("Login failed: Incorrect credentials")\n                        return False\n                    logger.info("Login successful!")\n                    return True\n            except Exception as e:\n                logger.error(f"Login attempt {attempt + 1} failed: {e}")\n                if attempt < CONFIG[\'max_retries\'] - 1:\n                    await asyncio.sleep(CONFIG[\'retry_delay\'])\n                else:\n         

In [358]:
sample_50["labor_transfer_label"] = labor_transfer_labels
sample_50.columns

Index(['conversation', 'timestamp', 'is_work_related', 'level_2_task',
       'level_1_task', 'level_0_task', 'Function', 'orig_index', 'interaction',
       'social_dimensions', 'role_sentence', 'role', 'role_core',
       'embedding_role_core', 'cluster_id', 'emotion', 'timestamp_local',
       'hour', 'title', 'labor_transfer_label'],
      dtype='object')

In [359]:
temp = sample_50[["conversation", "level_0_task", "title", "labor_transfer_label"]]
temp.to_csv("conversations_sample_50_with_labels.csv", index=False)

## Validation

This is gonna contain the validation of task-conversation mapping, Automation/augmentation labels and role extraction.


In [ ]:
work_related_labelled = pd.read_excel(WORK_RELATED_LABELED_DATA_PATH)
work_related_labelled["is_work_related"] = (
    work_related_labelled["is_work_related"].str.strip().str.capitalize()
)
work_related_labelled["correct"] = (
    work_related_labelled["correct"].str.strip().str.capitalize()
)

valid_mask = work_related_labelled["is_work_related"].isin(
    ["Yes", "No"]
) & work_related_labelled["correct"].isin(["Yes", "No"])

df_clean = work_related_labelled[valid_mask]

y_pred = df_clean["is_work_related"].values
y_true = df_clean["correct"].values

tp = sum((y_pred == "Yes") & (y_true == "Yes"))
tn = sum((y_pred == "No") & (y_true == "No"))
fp = sum((y_pred == "Yes") & (y_true == "No"))
fn = sum((y_pred == "No") & (y_true == "Yes"))

total_valid = len(df_clean)
accuracy = (tp + tn) / total_valid if total_valid > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0


print(f"Accuracy: {accuracy:.2%}")
print(f"False Positive Rate: {fpr:.2%}")
print(f"True Positive Rate: {tpr:.2%}\n\n")

print(pd.crosstab(y_true, y_pred, rownames=["Actual"], colnames=["Predicted"]))

Accuracy: 83.00%
False Positive Rate: 15.00%
True Positive Rate: 82.50%


Predicted  No  Yes
Actual            
No         17    3
Yes        14   66


In [131]:
len(work_related_labelled)

100

In [ ]:
validated_work_related_only = df_clean[
    (df_clean["correct"] == "Yes") & (df_clean["is_work_related"] == "Yes")
].copy()

In [ ]:
if not WORK_RELATED_ONLY_LABELED_DATA_PATH.exists():
    temp_path = OUTPUT_DIR / "validation_anthropic_aut_aug.csv"
    validation_anthropic_aut_aug = anthropic_automation_augmentation(
        conversations=validated_work_related_only[["conversation"]],
        path=temp_path,
    )

    validated_work_related_only = pd.merge(
        left=validated_work_related_only,
        right=validation_anthropic_aut_aug,
        left_on="conversation",
        right_on="conversation",
        how="inner",
    )

    validated_work_related_only.to_excel(
        WORK_RELATED_ONLY_LABELED_DATA_PATH, index=False
    )
else:
    validated_work_related_only = pd.read_excel(WORK_RELATED_ONLY_LABELED_DATA_PATH)
validated_work_related_only.head(1)

,Unnamed: 0,conversation,is_work_related,correct,anthropic_aut_aug_label,manual_auto_aug_label
0,1,"user: Make the first part of the Delta Systems history, I made the table’s first 2 parts\n“Progressbar Compass” Era:\nProgressbar Compass was released in 1999, with codename 98_compass (Internal). First part of the project, a heavy-modified Progressbar 98 SE.\nProgressbar Compass: Revision 2 was made in 2004, with codename dt_compass (Internal). Second part of the project.\nAlso a Wiki code that I used to make:\n=== Progressbar Compass ===\nTBA.\n\n=== Project Compass Development ===\nSome years later, it was speculated that a new system would appear in the market. An ex-employee of an unknown company revealed this system to 3 people, it is called “Project Compass”, the employee, whose name is Jacob Thompson, confirmed that the reason for this Beta code was because the system would be very easy to handle, and with few applications.\nUsing the table's parts, make the thing from the Project Compass Development\nassistant: era:\nIn response to the buzz around the rumored ""Project Compass"", Delta Systems began development on their own version of the system. In 2005, they announced the start of ""Project Compass Development"" with the goal of creating a user-friendly and efficient operating system with a sleek design. Delta Systems hired Jacob Thompson, the ex-employee who leaked information about the original ""Project Compass"", to consult on the project. The internal codename for this project was ""delta_compass"". The development process lasted for several years, with multiple revisions and updates. Delta Systems aimed to release the final version of the operating system in 2010.\n",Yes,Yes,\nDirective\n,Directive


In [ ]:
valid_labels = validated_work_related_only.dropna(
    subset=["anthropic_aut_aug_label", "manual_auto_aug_label"]
)
valid_labels["anthropic_aut_aug_label"] = (
    valid_labels["anthropic_aut_aug_label"].str.strip().str.lower()
)
valid_labels["manual_auto_aug_label"] = (
    valid_labels["manual_auto_aug_label"].str.strip().str.lower()
)

kappa = cohen_kappa_score(
    valid_labels["anthropic_aut_aug_label"], valid_labels["manual_auto_aug_label"]
)

print(f"Cohen's Kappa (Automation vs Augmentation): {kappa:.4f}")

Cohen's Kappa (Automation vs Augmentation): 0.5848


In [ ]:
if not TASK_MAPPING_LABELED_DATA_PATH.exists():
    temp_path = OUTPUT_DIR / "task_mapping_labeled.csv"
    validation_task_mapping = map_conversation_to_task(
        conversations=validated_work_related_only[["conversation"]],
        tasks=hierarchy_df,
        path=temp_path,
    )

    validated_work_related_only = pd.merge(
        left=validated_work_related_only,
        right=validation_task_mapping,
        left_on="conversation",
        right_on="conversation",
        how="inner",
    )

    validated_work_related_only.to_excel(TASK_MAPPING_LABELED_DATA_PATH, index=False)
else:
    validated_work_related_only = pd.read_excel(TASK_MAPPING_LABELED_DATA_PATH)
validated_work_related_only.head(1)

,Unnamed: 0,conversation,is_work_related,correct,anthropic_aut_aug_label,manual_auto_aug_label,level_2_task,level_1_task,level_0_task,task_plausible
0,1,"user: Make the first part of the Delta Systems history, I made the table’s first 2 parts\n“Progressbar Compass” Era:\nProgressbar Compass was released in 1999, with codename 98_compass (Internal). First part of the project, a heavy-modified Progressbar 98 SE.\nProgressbar Compass: Revision 2 was made in 2004, with codename dt_compass (Internal). Second part of the project.\nAlso a Wiki code that I used to make:\n=== Progressbar Compass ===\nTBA.\n\n=== Project Compass Development ===\nSome years later, it was speculated that a new system would appear in the market. An ex-employee of an unknown company revealed this system to 3 people, it is called “Project Compass”, the employee, whose name is Jacob Thompson, confirmed that the reason for this Beta code was because the system would be very easy to handle, and with few applications.\nUsing the table's parts, make the thing from the Project Compass Development\nassistant: era:\nIn response to the buzz around the rumored ""Project Compass"", Delta Systems began development on their own version of the system. In 2005, they announced the start of ""Project Compass Development"" with the goal of creating a user-friendly and efficient operating system with a sleek design. Delta Systems hired Jacob Thompson, the ex-employee who leaked information about the original ""Project Compass"", to consult on the project. The internal codename for this project was ""delta_compass"". The development process lasted for several years, with multiple revisions and updates. Delta Systems aimed to release the final version of the operating system in 2010.\n",Yes,Yes,\nDirective\n,Directive,"Assist with creative content, marketing, and media production projects","Assist with creative fiction writing, editing, and development","write original or adapted material for dramas, comedies, puppet shows, narration, or other performances.",Yes


In [ ]:
task_agreement = len(
    validated_work_related_only[validated_work_related_only["task_plausible"] == "Yes"]
) / len(validated_work_related_only)
task_agreement

0.8615384615384616

In [ ]:
if not ROLES_LABELED_DATA_PATH.exists():
    temp_path = OUTPUT_DIR / "roles_labeled.csv"
    validation_roles = assign_role(
        conversations=validated_work_related_only[["conversation"]], path=temp_path
    )

    validated_work_related_only = pd.merge(
        left=validated_work_related_only,
        right=validation_roles,
        left_on="conversation",
        right_on="conversation",
        how="inner",
    )

    validated_work_related_only.to_excel(ROLES_LABELED_DATA_PATH, index=False)
else:
    validated_work_related_only = pd.read_excel(ROLES_LABELED_DATA_PATH)
validated_work_related_only.head(1)

,Unnamed: 0,conversation,is_work_related,correct,anthropic_aut_aug_label,manual_auto_aug_label,level_2_task,level_1_task,level_0_task,task_plausible,llm_analysis
0,1,"user: Make the first part of the Delta Systems history, I made the table’s first 2 parts\n“Progressbar Compass” Era:\nProgressbar Compass was released in 1999, with codename 98_compass (Internal). First part of the project, a heavy-modified Progressbar 98 SE.\nProgressbar Compass: Revision 2 was made in 2004, with codename dt_compass (Internal). Second part of the project.\nAlso a Wiki code that I used to make:\n=== Progressbar Compass ===\nTBA.\n\n=== Project Compass Development ===\nSome years later, it was speculated that a new system would appear in the market. An ex-employee of an unknown company revealed this system to 3 people, it is called “Project Compass”, the employee, whose name is Jacob Thompson, confirmed that the reason for this Beta code was because the system would be very easy to handle, and with few applications.\nUsing the table's parts, make the thing from the Project Compass Development\nassistant: era:\nIn response to the buzz around the rumored ""Project Compass"", Delta Systems began development on their own version of the system. In 2005, they announced the start of ""Project Compass Development"" with the goal of creating a user-friendly and efficient operating system with a sleek design. Delta Systems hired Jacob Thompson, the ex-employee who leaked information about the original ""Project Compass"", to consult on the project. The internal codename for this project was ""delta_compass"". The development process lasted for several years, with multiple revisions and updates. Delta Systems aimed to release the final version of the operating system in 2010.\n",Yes,Yes,\nDirective\n,Directive,"Assist with creative content, marketing, and media production projects","Assist with creative fiction writing, editing, and development","write original or adapted material for dramas, comedies, puppet shows, narration, or other performances.",Yes,"[{'interaction': 'INTERACTION BETWEEN AI AND HUMAN', 'social_dimensions': ['Knowledge'], 'role_object': {'role': 'Knowledge Provider', 'role_sentence': ""This conversation represents an exchange of Knowledge, and in this context, the AI acts as a Knowledge Provider because it generates the historical content and fulfills the user's request for the first part of Delta Systems history.""}}, {'interaction': 'INTERACTION BETWEEN AI AND HUMAN', 'social_dimensions': ['Knowledge'], 'role_object': {'role': 'Knowledge Provider', 'role_sentence': 'This conversation represents an exchange of Knowledge, and in this context, the AI acts as a Knowledge Provider because it continues to supply the requested historical narrative based on user inputs.'}}]"


In [ ]:
if not VALIDATION_LABELED_DATA_PATH.exists():
    temp = extract_role_object(
        df=validated_work_related_only, column_name="llm_analysis"
    )
    validation_final_df = pd.merge(
        left=validated_work_related_only,
        right=temp,
        left_index=True,
        right_on="conversation_idx",
        how="inner",
    )
    validation_final_df.to_excel(VALIDATION_LABELED_DATA_PATH, index=False)
else:
    validation_final_df = pd.read_excel(VALIDATION_LABELED_DATA_PATH)
validation_final_df.head(1)

,Unnamed: 0,conversation,is_work_related,correct,anthropic_aut_aug_label,manual_auto_aug_label,level_2_task,level_1_task,level_0_task,task_plausible,llm_analysis,conversation_idx,role_sentence,role_label
0,1,"user: Make the first part of the Delta Systems history, I made the table’s first 2 parts\n“Progressbar Compass” Era:\nProgressbar Compass was released in 1999, with codename 98_compass (Internal). First part of the project, a heavy-modified Progressbar 98 SE.\nProgressbar Compass: Revision 2 was made in 2004, with codename dt_compass (Internal). Second part of the project.\nAlso a Wiki code that I used to make:\n=== Progressbar Compass ===\nTBA.\n\n=== Project Compass Development ===\nSome years later, it was speculated that a new system would appear in the market. An ex-employee of an unknown company revealed this system to 3 people, it is called “Project Compass”, the employee, whose name is Jacob Thompson, confirmed that the reason for this Beta code was because the system would be very easy to handle, and with few applications.\nUsing the table's parts, make the thing from the Project Compass Development\nassistant: era:\nIn response to the buzz around the rumored ""Project Compass"", Delta Systems began development on their own version of the system. In 2005, they announced the start of ""Project Compass Development"" with the goal of creating a user-friendly and efficient operating system with a sleek design. Delta Systems hired Jacob Thompson, the ex-employee who leaked information about the original ""Project Compass"", to consult on the project. The internal codename for this project was ""delta_compass"". The development process lasted for several years, with multiple revisions and updates. Delta Systems aimed to release the final version of the operating system in 2010.\n",Yes,Yes,\nDirective\n,Directive,"Assist with creative content, marketing, and media production projects","Assist with creative fiction writing, editing, and development","write original or adapted material for dramas, comedies, puppet shows, narration, or other performances.",Yes,"[{'interaction': 'INTERACTION BETWEEN AI AND HUMAN', 'social_dimensions': ['Knowledge'], 'role_object': {'role': 'Knowledge Provider', 'role_sentence': ""This conversation represents an exchange of Knowledge, and in this context, the AI acts as a Knowledge Provider because it generates the historical content and fulfills the user's request for the first part of Delta Systems history.""}}, {'interaction': 'INTERACTION BETWEEN AI AND HUMAN', 'social_dimensions': ['Knowledge'], 'role_object': {'role': 'Knowledge Provider', 'role_sentence': 'This conversation represents an exchange of Knowledge, and in this context, the AI acts as a Knowledge Provider because it continues to supply the requested historical narrative based on user inputs.'}}]",0,"This conversation represents an exchange of Knowledge, and in this context, the AI acts as a Knowledge Provider because it generates the historical content and fulfills the user's request for the first part of Delta Systems history.",Yes


In [145]:
validation_final_df.shape

(165, 14)

In [ ]:
role_agreement = len(
    validation_final_df[validation_final_df["role_label"] == "Yes"]
) / len(validation_final_df)
print(f"Role agreement: {role_agreement:.2%}")

Role agreement: 85.45%
